In [1]:
import pandas as pd
import numpy as np
import pickle
import SimpleITK as sitk
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
from iblatlas.regions import BrainRegions
import torch
import torch.nn as nn
import torch.optim as optim
import random
from tqdm import tqdm
import torch.nn.functional as F
from torch.optim import lr_scheduler
import time

In [2]:
df_layers = pd.read_excel('SEU_net_w_corticalLayers.xlsx', header=1, index_col=1)
df_layers = df_layers.iloc[:, 1:]
df_layers

,FRP2/3,MOp1,MOp2/3,MOp5,MOp6a,MOs1,MOs2/3,MOs5,MOs6a,SSp-n2/3,...,MD,SMT,PVT,RE,CM,RT,PSV,PB,TRN,PRNr
FRP2/3,0.000000,0.0,0.000000,0.000000,0.000000,0.835160,2.300365,1.791165,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.000000,0,0,0,0.000000
MOp1,0.000000,0.0,1.435472,2.390881,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.000000,0,0,0,0.000000
MOp2/3,0.625221,0.0,2.078293,2.099848,1.924731,0.715541,1.498042,1.575019,0.0,0.426072,...,0.000000,0.00000,0.0,0.000000,0.0,0.932975,0,0,0,0.875238
MOp5,0.000000,0.0,1.068048,3.553463,1.668447,0.409365,2.183911,2.279111,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.794181,0,0,0,0.000000
MOp6a,0.000000,0.0,2.304936,0.488099,2.354786,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.917643,0,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RT,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,1.514522,2.13142,0.0,4.582369,0.0,0.762490,0,0,0,0.000000
PSV,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.000000,0,0,0,0.720976
PB,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.000000,0,0,0,1.214098
TRN,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.00000,0.0,0.000000,0.0,0.000000,0,0,0,0.000000


In [3]:
with open('brain_region_w_layer_stage2.pkl', 'rb') as f:
    C_traces = pickle.load(f)
C_traces

{'MG': {'spike_times': array([1.32295741e-06, 2.22448016e-06, 4.46894026e-06, ...,
         9.99998459e-01, 9.99998823e-01, 1.00000000e+00]),
  'spike_voltages': array([1.02225975e-04, 1.90227904e-04, 6.86466592e-05, ...,
         1.46565464e-04, 1.36938193e-04, 8.16656393e-05])},
 'LGd': {'spike_times': array([0., 0., 0., ..., 1., 1., 1.]),
  'spike_voltages': array([1.15912066e-04, 2.21375594e-04, 2.84903537e-04, ...,
         7.53099230e-05, 1.34822039e-04, 6.84298866e-05])},
 'CEA': {'spike_times': array([9.88512933e-07, 3.90553693e-06, 7.73567368e-06, ...,
         9.99995064e-01, 9.99997947e-01, 9.99999048e-01]),
  'spike_voltages': array([2.71024008e-04, 1.77588983e-04, 8.60513755e-05, ...,
         1.29396503e-04, 9.81526682e-05, 5.01546710e-05])},
 'VPM': {'spike_times': array([4.29821969e-10, 5.48716916e-10, 1.60778299e-09, ...,
         1.00000000e+00, 1.00000000e+00, 1.00000000e+00]),
  'spike_voltages': array([0.00018025, 0.00016094, 0.00014604, ..., 0.00022785, 0.00015073

In [4]:
keys = list(C_traces.keys())

df_filtered = df_layers.loc[df_layers.index.intersection(keys), df_layers.columns.intersection(keys)]
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_filtered),
    index=df_filtered.index,
    columns=df_filtered.columns
)
df_scaled

,FRP2/3,MOp1,MOp2/3,MOp5,MOp6a,MOs1,MOs2/3,MOs5,MOs6a,SSp-n2/3,...,LD,MD,SMT,PVT,CM,RT,PSV,PB,TRN,PRNr
FRP2/3,0.000000,0.0,0.000000,0.000000,0.000000,0.331930,0.421616,0.600585,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
MOp1,0.000000,0.0,0.157859,0.293038,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
MOp2/3,0.413603,0.0,0.228551,0.257367,0.643928,0.284388,0.274564,0.528111,0.0,0.028615,...,0.0,0.000000,0.000000,0.0,0.0,0.403550,0.0,0.0,0.0,0.720896
MOp5,0.000000,0.0,0.117454,0.435530,0.558187,0.162700,0.400272,0.764195,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.343516,0.0,0.0,0.0,0.000000
MOp6a,0.000000,0.0,0.253475,0.059824,0.787805,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.396918,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RT,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,1.0,0.477782,0.274014,0.0,0.0,0.329808,0.0,0.0,0.0,0.000000
PSV,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.593837
PB,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,1.000000
TRN,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000


In [5]:
df_scaled.to_csv("./result/w_layer/stage2/df_scaled.csv", index=True)

In [6]:
def build_region_binned_traces_nan(aligned_region_data,
                                   dt=0.01,
                                   duration_mode='percentile',
                                   duration_percentile=100):

    end_times = []
    for rid, d in aligned_region_data.items():
        t = d.get('spike_times', None)
        if t is None or len(t) == 0:
            end_times.append(0.0)
        else:
            end_times.append(float(np.max(t)))
    end_times = np.array(end_times)

    if duration_mode == 'min':
        used_duration = float(np.min(end_times))
    elif duration_mode == 'median':
        used_duration = float(np.median(end_times))
    elif duration_mode == 'percentile':
        used_duration = float(np.percentile(end_times, duration_percentile))
    elif duration_mode == 'max':
        used_duration = float(np.max(end_times))
    else:
        raise ValueError("duration_mode must be 'min'|'median'|'percentile'|'max'")

    if used_duration <= 0:
        raise ValueError("used_duration <= 0, check your data")

    edges = np.arange(0.0, used_duration + dt, dt)
    centers = edges[:-1] + dt / 2.0
    n_bins = len(centers)

    binned = {}
    for rid, d in aligned_region_data.items():
        times = np.asarray(d.get('spike_times', []))
        volts = d.get('spike_voltages', None)

        region_bin = np.full(n_bins, np.nan, dtype=float)

        if times.size == 0:
            binned[rid] = region_bin
            continue

        mask = (times >= 0) & (times < used_duration)
        times_cut = times[mask]
        if volts is not None:
            volts_cut = np.asarray(volts)[mask]
        else:
            volts_cut = None

        bin_indices = np.digitize(times_cut, edges) - 1
        valid = (bin_indices >= 0) & (bin_indices < n_bins)
        bin_indices = bin_indices[valid]
        if volts_cut is not None:
            volts_cut = volts_cut[valid]
            sums = np.zeros(n_bins)
            counts = np.zeros(n_bins)
            np.add.at(sums, bin_indices, volts_cut)
            np.add.at(counts, bin_indices, 1)
            avg = np.divide(sums, counts, out=np.full_like(sums, np.nan), where=counts > 0)
            region_bin = avg
        else:
            counts = np.zeros(n_bins)
            np.add.at(counts, bin_indices, 1)
            region_bin[counts > 0] = counts[counts > 0]  

        binned[rid] = region_bin

    return centers, binned, used_duration


#################################################
C_traces_sorted = {k: C_traces[k] for k in df_scaled.index if k in C_traces}
time_axis, binned_traces, duration = build_region_binned_traces_nan(C_traces_sorted, dt=0.001)

In [7]:
# ----------------- Seed -----------------
def set_seed(seed=42):
    import random, os
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# ----------------- Graph-leaning -----------------
class SimpleGraphLearn(nn.Module):
    def __init__(self, node_dim, nodes, dropout=0.5, struct_connectome=None):
        super(SimpleGraphLearn, self).__init__()
        self.nodes = nodes
        self.dropout = dropout
        
        self.gate_net = nn.Sequential(
            nn.Linear(node_dim * 2, node_dim),
            nn.Sigmoid()
        )
        
        self.attention = nn.Linear(node_dim, 1, bias=False)
        self.norm = nn.LayerNorm(nodes)
        
        self.w = nn.Parameter(torch.zeros(size=(nodes, node_dim)))
        nn.init.xavier_uniform_(self.w.data, gain=1.414)
        
        if struct_connectome is not None:
            self.struct_connectome = nn.Parameter(
                torch.FloatTensor(struct_connectome), 
                requires_grad=False
            )
        else:
            self.struct_connectome = None
            
        self.graph_weight = nn.Parameter(torch.ones(1))
        self.struct_weight = nn.Parameter(torch.ones(1))
        
    def forward(self, node_features, temporal_features=None, batch_size=1):
        batch_size, nodes, node_dim = node_features.shape
        
        if temporal_features is not None:
            concat_features = torch.cat([node_features, temporal_features], dim=-1)  # (B, N, 2*D)
            gate = self.gate_net(concat_features)  # (B, N, D)
            node_features = gate * node_features + (1 - gate) * temporal_features
        
        node_res = torch.einsum('bnd, nl -> bnl', node_features, self.w) + node_features
        skip_atten = torch.einsum('bnd,bdm->bnm', node_res, node_res.transpose(-1, -2))
        skip_atten = self.norm(skip_atten)
        
        attention_scores = torch.matmul(node_features, node_features.transpose(-2, -1))  # (B, N, N)
        attention_scores = attention_scores / (node_dim ** 0.5)
        attention_scores = F.dropout(attention_scores, self.dropout, training=self.training)
        
        adj_raw = attention_scores + skip_atten
        
        if self.struct_connectome is not None:
            struct_adj = self.struct_connectome.unsqueeze(0).expand(batch_size, -1, -1)
            adj_dynamic = self.graph_weight * adj_raw + self.struct_weight * struct_adj
        else:
            adj_dynamic = adj_raw
            
        adj_dynamic = torch.tanh(adj_dynamic)
        
        if self.struct_connectome is not None:
            struct_upper_bound = self.struct_connectome.unsqueeze(0).expand(batch_size, -1, -1)
            adj_dynamic = torch.clamp(adj_dynamic, -struct_upper_bound, struct_upper_bound)
        
        return adj_dynamic

# ----------------- Temporal dynamic graph learning model -----------------
class TimeSeriesDynamicGraphModel(nn.Module):
    def __init__(self, device, num_nodes, seq_length=12, in_dim=1, out_dim=1, 
                 hidden_dim=32, dropout=0.3, embed_dim=10, heads=4, head_dim=8, 
                 struct_connectome=None):
        super(TimeSeriesDynamicGraphModel, self).__init__()
        self.device = device
        self.num_nodes = num_nodes
        self.seq_length = seq_length
        self.hidden_dim = hidden_dim
        self.dropout = dropout

        self.node_embedding = nn.Embedding(num_nodes, embed_dim)
        
        self.temporal_fusion = nn.Linear(hidden_dim, embed_dim)
        
        self.graph_learn = SimpleGraphLearn(
            node_dim=embed_dim, 
            nodes=num_nodes,
            dropout=dropout,
            struct_connectome=struct_connectome
        )

        self.temporal_encoder = nn.LSTM(
            input_size=in_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True,
            dropout=dropout
        )

        self.gcn_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(2)
        ])

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, input, pred_time_embed=None):
        batch_size, seq_length, num_nodes, in_dim = input.shape
        
        idx = torch.arange(num_nodes).to(self.device)
        node_embeddings = self.node_embedding(idx)  # (num_nodes, embed_dim)
        node_embeddings = node_embeddings.unsqueeze(0).expand(batch_size, -1, -1)  # (batch_size, num_nodes, embed_dim)
        
        x_reshaped = input.permute(0, 2, 1, 3).contiguous().view(batch_size * num_nodes, seq_length, in_dim)
        temporal_out, _ = self.temporal_encoder(x_reshaped)
        
        temporal_features = temporal_out.view(batch_size, num_nodes, seq_length, -1)
        
        dynamic_graphs = []
        temporal_hiddens = []
        
        for t in range(seq_length):
            temporal_features_t = temporal_features[:, :, t, :]  # (batch_size, num_nodes, hidden_dim)
            
            temporal_features_proj_t = self.temporal_fusion(temporal_features_t)  # (batch_size, num_nodes, embed_dim)
            
            adj_dynamic_t = self.graph_learn(
                node_embeddings, 
                temporal_features_proj_t, 
                batch_size
            )  # (batch_size, num_nodes, num_nodes)
            
            dynamic_graphs.append(adj_dynamic_t)
            temporal_hiddens.append(temporal_features_t)
        
        adj_dynamic_all = torch.stack(dynamic_graphs, dim=1)  # (batch_size, seq_length, num_nodes, num_nodes)
        
        gcn_outputs_all = []
        
        for t in range(seq_length):
            h_t = temporal_hiddens[t]  # (batch_size, num_nodes, hidden_dim)
            adj_t = dynamic_graphs[t]  # (batch_size, num_nodes, num_nodes)
            
            gcn_outputs_t = []
            current_h = h_t
            for gcn_layer in self.gcn_layers:
                h_gcn = torch.matmul(adj_t, current_h)  
                h_gcn = gcn_layer(h_gcn)
                h_gcn = F.relu(h_gcn)
                h_gcn = F.dropout(h_gcn, self.dropout, training=self.training)
                gcn_outputs_t.append(h_gcn)
                current_h = h_gcn
            
            gcn_outputs_all.append(current_h)
        
        gcn_outputs_all = torch.stack(gcn_outputs_all, dim=1)  # (batch_size, seq_length, num_nodes, hidden_dim)
        
        final_hidden = gcn_outputs_all[:, -1, :, :]  # (batch_size, num_nodes, hidden_dim)
        output = self.output_layer(final_hidden)  # (batch_size, num_nodes, out_dim)
        
        return output, None, adj_dynamic_all

# ----------------- Trainer -----------------
class trainer():
    def __init__(self, num_nodes, seq_length, in_dim, out_dim, hidden_dim, dropout, lrate, wdecay, device, 
                 embed_dim, heads=4, head_dim=8, struct_connectome=None, struct_constraint_weight=0.2):
        
        self.model = TimeSeriesDynamicGraphModel(
            device, num_nodes, seq_length, in_dim, out_dim,
            hidden_dim=hidden_dim, dropout=dropout, embed_dim=embed_dim,
            heads=heads, head_dim=head_dim, struct_connectome=struct_connectome
        )
        
        self.model.to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lrate, weight_decay=wdecay)
        self.scheduler = lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=10, eps=0.00001, cooldown=20, verbose=True
        )
        
        self.loss = nn.MSELoss()
        self.clip = 0.5
        self.struct_constraint_weight = struct_constraint_weight

    def train(self, input, real_val, iter=0):
        self.model.train()
        output, gl_loss, adj_d = self.model(input)
        
        loss = self.loss(output, real_val)
        
        if hasattr(self.model.graph_learn, 'struct_connectome') and \
           self.model.graph_learn.struct_connectome is not None:
            
            # adj_d (batch_size, seq_length, num_nodes, num_nodes)
            if len(adj_d.shape) == 4:  # (batch_size, seq_length, num_nodes, num_nodes)
                avg_adj = torch.mean(adj_d, dim=(0, 1))  
            elif len(adj_d.shape) == 3:  # (batch_size, num_nodes, num_nodes)
                avg_adj = torch.mean(adj_d, dim=0)
            else:
                avg_adj = adj_d
            
            struct_adj = self.model.graph_learn.struct_connectome
            
            avg_adj_abs = torch.abs(avg_adj)
            struct_loss = F.mse_loss(avg_adj_abs, struct_adj)
            
            loss = loss + self.struct_constraint_weight * struct_loss
        
        loss.backward()
        
        if self.clip is not None:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip)
        
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        return loss.item()

    def eval(self, input, real_val):
        self.model.eval()
        with torch.no_grad():
            output, gl_loss, adj_d = self.model(input)
            loss = self.loss(output, real_val)
        return loss.item(), output, adj_d

# ----------------- Main training function -----------------
def train_timeseries_dynamic_graph(
    cell_trace, struct_connectome, seq_length, pred_length,
    hidden_dim, lr, epochs, batch_size, dropout,
    device=None, seed=42, val_ratio=0.2, patience=20,
    heads=4, head_dim=8, embed_dim=10, struct_constraint_weight=0.5,
    save_dir="./results/wo_layer/stage1"   
):

    import os, numpy as np, torch, random
    from tqdm import tqdm

    set_seed(seed)
    # Create save folder
    os.makedirs(save_dir, exist_ok=True)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    num_regions, total_time = cell_trace.shape
    if seq_length is None:
        seq_length = total_time - pred_length

    # ------------------ Data input ------------------
    data, targets = [], []
    cell_trace = (cell_trace - np.mean(cell_trace)) / (np.std(cell_trace) + 1e-8)

    for t in range(total_time - seq_length - pred_length + 1):
        x = cell_trace[:, t:t+seq_length].T.reshape(seq_length, num_regions, 1)
        y = cell_trace[:, t+seq_length:t+seq_length+pred_length].T.reshape(num_regions, pred_length)
        data.append(x); targets.append(y)

    data, targets = np.array(data), np.array(targets)
    split = int(len(data) * (1 - val_ratio))

    train_data = torch.FloatTensor(data[:split]).to(device)
    train_targets = torch.FloatTensor(targets[:split]).to(device)
    val_data = torch.FloatTensor(data[split:]).to(device)
    val_targets = torch.FloatTensor(targets[split:]).to(device)

    # ------------------ Model ------------------
    engine = trainer(
        num_nodes=num_regions, seq_length=seq_length, in_dim=1, out_dim=pred_length,
        hidden_dim=hidden_dim, dropout=dropout, lrate=lr, wdecay=0.0, device=device,
        embed_dim=embed_dim, heads=heads, head_dim=head_dim,
        struct_connectome=struct_connectome, struct_constraint_weight=struct_constraint_weight
    )

    best_val = float('inf'); patience_count = 0
    train_losses, val_losses = [], []
    epoch_avg_graphs = []

    print(f"Training start | seq={seq_length}, T={total_time}")

    # ------------------ Train ------------------
    for epoch in range(epochs):
        engine.model.train()
        train_loss = 0
        for i in range(0, len(train_data), batch_size):
            train_loss += engine.train(train_data[i:i+batch_size], train_targets[i:i+batch_size])
        train_loss /= (len(train_data)//batch_size + 1)
        train_losses.append(train_loss)

        # -------- Validation --------
        engine.model.eval()
        val_loss = 0
        with torch.no_grad():
            for i in range(0, len(val_data), batch_size):
                val_loss += engine.eval(val_data[i:i+batch_size], val_targets[i:i+batch_size])[0]
        val_loss /= (len(val_data)//batch_size + 1)
        val_losses.append(val_loss)
        engine.scheduler.step(val_loss)

        # ---- Save best ----
        if val_loss < best_val:
            best_val = val_loss
            patience_count = 0
            torch.save(engine.model.state_dict(), f"{save_dir}/best_model.pth")  
        else:
            patience_count += 1
        if patience_count >= patience:
            print(f"Early stop @ epoch {epoch+1}")
            break

        # ---- collect epoch avg graph ----
        with torch.no_grad():
            x = train_data[0:1]
            _, _, adj = engine.model(x)
            epoch_avg_graphs.append(np.mean(adj.squeeze(0).cpu().numpy(), axis=0))

        if (epoch+1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train {train_loss:.5f} | Val {val_loss:.5f}")

    print("Training done, loading best model")
    engine.model.load_state_dict(torch.load(f"{save_dir}/best_model.pth"))
    engine.model.eval()

    np.save(f"{save_dir}/train_losses.npy", np.array(train_losses))
    np.save(f"{save_dir}/val_losses.npy", np.array(val_losses))
    np.save(f"{save_dir}/epoch_avg_graphs.npy", np.array(epoch_avg_graphs))
    print(f"Saved epoch_avg_graphs.npy to {save_dir}")

    # ------------------ Full time inference ------------------
    print("Reconstructing full dynamic graph...")

    full_dynamic_graphs = np.zeros((total_time, num_regions, num_regions))
    frame_count = np.zeros(total_time)

    with torch.no_grad():
        for t in tqdm(range(total_time - seq_length + 1)):
            x = cell_trace[:, t:t+seq_length].T.reshape(1, seq_length, num_regions, 1)
            x = torch.FloatTensor(x).to(device)

            _, _, adj_seq = engine.model(x)
            adj_seq = adj_seq.squeeze(0).cpu().numpy()

            for k in range(seq_length):
                idx = t + k
                full_dynamic_graphs[idx] += adj_seq[k]
                frame_count[idx] += 1

    for i in range(total_time):
        if frame_count[i] > 0:
            full_dynamic_graphs[i] /= frame_count[i]

    np.save(f"{save_dir}/dynamic_full_graphs.npy", full_dynamic_graphs)
    print(f"Saved dynamic_full_graphs.npy to {save_dir}, shape={full_dynamic_graphs.shape}")

    return engine.model, train_losses, val_losses, full_dynamic_graphs

In [8]:
struct_connectome_array = df_scaled.values
binned_traces_array = np.array([binned_traces[key] for key in sorted(binned_traces.keys())])
binned_traces_array = np.nan_to_num(binned_traces_array, nan=0.0)

import itertools
import os

# ====== Hyperparam grids ======
#batch_sizes = [8, 16, 32, 64]
batch_sizes = [64]
dropouts = [0.1, 0.3, 0.6, 0.8]
struct_weights = [0.1, 0.5, 0.7, 1.0]

# ====== Loop over all combinations ======
for bs, dp, sw in itertools.product(batch_sizes, dropouts, struct_weights):

    # 生成保存目录名称：比如 bs32_dp0.6_sw0.5
    save_dir = f"./result/w_layer/stage2/tuning/bs{bs}_dp{dp}_sw{sw}"
    os.makedirs(save_dir, exist_ok=True)

    print("="*60)
    print(f"Training with batch_size={bs}, dropout={dp}, struct_constraint_weight={sw}")
    print(f"Save dir: {save_dir}")
    print("="*60)

    model, train_losses, val_losses, full_dynamic_graphs = train_timeseries_dynamic_graph(
        cell_trace = binned_traces_array,
        struct_connectome = struct_connectome_array,
        seq_length = 50,
        pred_length = 10,
        hidden_dim = 32,
        lr = 1e-3,
        epochs = 300,
        batch_size = bs,
        dropout = dp,
        device = None,
        seed = 42,
        val_ratio = 0.2,
        patience = 20,   
        heads = 4,
        head_dim = 8,
        embed_dim = 10,
        struct_constraint_weight = sw,
        save_dir = save_dir
    )

    # 再保存曲线
    np.save(f"{save_dir}/train_losses.npy", np.array(train_losses))
    np.save(f"{save_dir}/val_losses.npy", np.array(val_losses))

    print(f"Finished: bs={bs} dp={dp} sw={sw} | Saved to {save_dir}")

Training with batch_size=64, dropout=0.1, struct_constraint_weight=0.1
Save dir: ./result/w_layer/stage2/tuning/bs64_dp0.1_sw0.1


/home/ylx/anaconda3/envs/ibl_bwm/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training start | seq=50, T=1000
Epoch 20/300 | Train 0.82036 | Val 0.86199
Epoch 40/300 | Train 0.76966 | Val 0.79580
Epoch 60/300 | Train 0.74778 | Val 0.77216
Epoch 80/300 | Train 0.73760 | Val 0.76337
Epoch 100/300 | Train 0.73247 | Val 0.75754
Epoch 120/300 | Train 0.72773 | Val 0.75405
Epoch 140/300 | Train 0.72380 | Val 0.75156
Epoch 160/300 | Train 0.72261 | Val 0.74968
Epoch 180/300 | Train 0.72015 | Val 0.74777
Epoch 200/300 | Train 0.71830 | Val 0.74670
Epoch 220/300 | Train 0.71687 | Val 0.74648
Epoch 240/300 | Train 0.71681 | Val 0.74569
Epoch 260/300 | Train 0.71502 | Val 0.74490
Epoch 280/300 | Train 0.71345 | Val 0.74484


/tmp/ipykernel_2995452/996028222.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  engine.model.load_state_dict(torch.load(f"{save_dir}/best_model.pth"))


Early stop @ epoch 299
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/w_layer/stage2/tuning/bs64_dp0.1_sw0.1
Reconstructing full dynamic graph...


100%|███████████████████████████████████████████████████████████████████████████████████████| 951/951 [00:26<00:00, 35.30it/s]


... (27 outputs left out)

100%|███████████████████████████████████████████████████████████████████████████████████████| 951/951 [00:26<00:00, 35.25it/s]


Saved dynamic_full_graphs.npy to ./result/w_layer/stage2/tuning/bs64_dp0.8_sw0.7, shape=(1000, 138, 138)
Finished: bs=64 dp=0.8 sw=0.7 | Saved to ./result/w_layer/stage2/tuning/bs64_dp0.8_sw0.7
Training with batch_size=64, dropout=0.8, struct_constraint_weight=1.0
Save dir: ./result/w_layer/stage2/tuning/bs64_dp0.8_sw1.0
Training start | seq=50, T=1000
Epoch 20/300 | Train 0.92279 | Val 0.99220
Epoch 40/300 | Train 0.92242 | Val 0.99214
Epoch 60/300 | Train 0.92183 | Val 0.99182
Epoch 80/300 | Train 0.91749 | Val 0.98793
Epoch 100/300 | Train 0.91585 | Val 0.98526
Epoch 120/300 | Train 0.91474 | Val 0.98449
Epoch 140/300 | Train 0.91422 | Val 0.98373
Early stop @ epoch 157
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/w_layer/stage2/tuning/bs64_dp0.8_sw1.0
Reconstructing full dynamic graph...


100%|███████████████████████████████████████████████████████████████████████████████████████| 951/951 [00:27<00:00, 34.48it/s]

Saved dynamic_full_graphs.npy to ./result/w_layer/stage2/tuning/bs64_dp0.8_sw1.0, shape=(1000, 138, 138)
Finished: bs=64 dp=0.8 sw=1.0 | Saved to ./result/w_layer/stage2/tuning/bs64_dp0.8_sw1.0
